In [7]:
!pip install mat73
!pip install --upgrade gspread

In [8]:
from google.colab import drive

drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
!lscpu |grep 'Model name'

Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz


In [10]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [11]:
!ls "/content/drive/My Drive/Ali AI IDS (1)/matlab/ftrain";

folder_name = "/content/drive/My Drive/Ali AI IDS (1)/matlab/ftrain/";

desktop.ini			  merge_00a0_ftrain_1x8.mat
dosAttack_0002_ftrain_1x8.mat	  merge_00a1_ftrain_1x8.mat
dosAttack_00a0_ftrain_1x8.mat	  merge_0130_ftrain_1x8.mat
dosAttack_00a1_ftrain_1x8.mat	  merge_0131_ftrain_1x8.mat
dosAttack_0130_ftrain_1x8.mat	  merge_0140_ftrain_1x8.mat
dosAttack_0131_ftrain_1x8.mat	  merge_0153_ftrain_1x8.mat
dosAttack_0140_ftrain_1x8.mat	  merge_018f_ftrain_1x8.mat
dosAttack_0153_ftrain_1x8.mat	  merge_01f1_ftrain_1x8.mat
dosAttack_018f_ftrain_1x8.mat	  merge_0260_ftrain_1x8.mat
dosAttack_01f1_ftrain_1x8.mat	  merge_02a0_ftrain_1x8.mat
dosAttack_0260_ftrain_1x8.mat	  merge_02b0_ftrain_1x8.mat
dosAttack_02a0_ftrain_1x8.mat	  merge_02c0_ftrain_1x8.mat
dosAttack_02c0_ftrain_1x8.mat	  merge_0316_ftrain_1x8.mat
dosAttack_0316_ftrain_1x8.mat	  merge_0329_ftrain_1x8.mat
dosAttack_0329_ftrain_1x8.mat	  merge_0350_ftrain_1x8.mat
dosAttack_0350_ftrain_1x8.mat	  merge_0370_ftrain_1x8.mat
dosAttack_0370_ftrain_1x8.mat	  merge_0430_ftrain_1x8.mat
dosAttack_0430_ftrain_1x8.mat	

In [14]:
import tensorflow as tf
from sklearn.metrics import confusion_matrix
from google.colab import auth
import gspread
from google.auth import default


# -----------------------------------------------------------
# FUNCTION: model_results
# INPUTS:
# model       : trained ML/DL model
# index       : row index for writing results in sheet
# attackType  : determines which worksheet to use
# -----------------------------------------------------------
def model_results(model, index, attackType):

    # -------------------------------------------------------
    # STEP 1: Model Prediction (Binary Classification assumed)
    # -------------------------------------------------------
    y_pred = tf.round(model.predict(X_test))   # Round probabilities → 0 or 1

    # -------------------------------------------------------
    # STEP 2: Compute Evaluation Metrics
    # -------------------------------------------------------

    # Accuracy
    acc = tf.keras.metrics.Accuracy()
    acc.update_state(y_test, y_pred)

    # Precision
    precision = tf.keras.metrics.Precision()
    precision.update_state(y_test, y_pred)

    # Recall
    recall = tf.keras.metrics.Recall()
    recall.update_state(y_test, y_pred)

    # Confusion Matrix Components
    fp = tf.keras.metrics.FalsePositives()
    fn = tf.keras.metrics.FalseNegatives()
    tp = tf.keras.metrics.TruePositives()
    tn = tf.keras.metrics.TrueNegatives()

    fp.update_state(y_test, y_pred)
    fn.update_state(y_test, y_pred)
    tp.update_state(y_test, y_pred)
    tn.update_state(y_test, y_pred)

    # -------------------------------------------------------
    # STEP 3: Authenticate Google Sheets Access
    # -------------------------------------------------------
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)

    # Open Google Sheet using key
    workbook = gc.open_by_key('1gtFRPRcp9gIW0_G1oZU562jk-XgUfuGXOcGOkiHvcvk')

    # -------------------------------------------------------
    # STEP 4: Select Worksheet Based on Attack Type
    # -------------------------------------------------------
    if attackType == 0:
        worksheet = workbook.worksheet('FuzzyAttack')
    elif attackType == 1:
        worksheet = workbook.worksheet('DoS')
    elif attackType == 2:
        worksheet = workbook.worksheet('Gear')
    elif attackType == 3:
        worksheet = workbook.worksheet('RPM')
    else:
        raise ValueError("Invalid attackType")

    # -------------------------------------------------------
    # STEP 5: Add Header (Title Row) IF first row
    # -------------------------------------------------------
    if index == 0:
        headers = [["Dataset", "FP", "FN", "TP", "TN",
                    "Accuracy", "Precision", "Recall"]]
        worksheet.update("A1", headers)

    # -------------------------------------------------------
    # STEP 6: Prepare Data Row
    # -------------------------------------------------------
    data_row = [[
        data_name,                                   # Dataset name
        float(fp.result().numpy()),
        float(fn.result().numpy()),
        float(tp.result().numpy()),
        float(tn.result().numpy()),
        float(acc.result().numpy()),
        float(precision.result().numpy()),
        float(recall.result().numpy())
    ]]

    # -------------------------------------------------------
    # STEP 7: Write Data to Sheet
    # -------------------------------------------------------
    row_position = "A" + str(index + 2)  # +2 because row 1 = header
    worksheet.update(row_position, data_row)

    # -------------------------------------------------------
    # STEP 8: Increment index (optional return)
    # -------------------------------------------------------
    return index + 1

In [ ]:
import mat73
import scipy.io
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
from sklearn.utils import validation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense
from tensorflow.keras import Sequential


for ii in range(0,26):
  canID = ['0002','00a0','00a1','0130','0131','0140','0153','018f','01f1','0260','02a0','02c0','0316','0329','0350','0370','0430','043f','0440','04b1','04f0','0545','05a0','05a2','05f0','0690']
  train_batch_size = 64
  test_batch_size = 64
  epoch_num= 5
  data_name = canID[ii]

# Load train dataset (merged data)
  data = mat73.loadmat(folder_name +'merge_'+ canID[ii] +'_ftrain_1x8.mat')


  label = data['labels']
  dataIn = data['inputMatrix']

  X_train, X_test, y_train, y_test = train_test_split(dataIn, label, test_size=0.3, random_state=42, shuffle=False)


 #ANN model structure
  tf.random.set_seed(42)
  model_single_neuron = Sequential([Dense(1, activation="sigmoid")])

  model_single_neuron.compile(loss="binary_crossentropy", optimizer=Adam(), metrics=["accuracy"])
  history = model_single_neuron.fit(X_train, y_train, epochs=epoch_num, batch_size=train_batch_size)


#saved weights
  name='Weights_'+data_name + '_' + str(epoch_num) + '_' + str(train_batch_size) + '_' +str(test_batch_size)
  model_single_neuron.save(folder_name+'weights/' + name)

  dataAttackNames = ['fuzzyAttack','dosAttack','gearSpoofing','rpmSpoofing']

  for attackType in range(0,4):
    dataAttackName = dataAttackNames[attackType]

    dataTest = mat73.loadmat(folder_name+dataAttackName+'_'+canID[ii] +'_ftrain_1x8.mat')
    dataTestIn = dataTest['inputMatrix']
    dataTestLabel = dataTest['labels']
    X_dummy_train, X_test, y_dummy_train, y_test = train_test_split(dataTestIn, dataTestLabel, test_size=0.3, random_state=42, shuffle=False)

    model_single_neuron.evaluate(X_test, y_test, batch_size=test_batch_size)
    model_results(model_single_neuron,ii,attackType)



Epoch 1/5
8464/8464 ━━━━━━━━━━━━━━━━━━━━ 22s 3ms/step - accuracy: 1.0000 - loss: 0.0026
Epoch 2/5
8464/8464 ━━━━━━━━━━━━━━━━━━━━ 16s 2ms/step - accuracy: 1.0000 - loss: 2.9989e-04
Epoch 3/5
8464/8464 ━━━━━━━━━━━━━━━━━━━━ 16s 2ms/step - accuracy: 1.0000 - loss: 1.3891e-04
Epoch 4/5
1730/8464 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - accuracy: 1.0000 - loss: 1.6825e-05